# 1.3 million mouse brain cells (scBIOT 1.2.0)

The input is downloaded from the original 10x Genomics source. The default reads a deterministic 20,000-cell sample directly from the compressed matrix so the notebook can run without first materializing all 1.3 million cells; set `SCBIOT_TUTORIAL_MAX_CELLS=0` for the complete analysis. The same v1.2.0 **linear autoencoder → OT** calls work on CPU or GPU.

- Original data: [10x 1.3 Million Brain Cells](https://www.10xgenomics.com/datasets/1-3-million-brain-cells-from-e18-mice-2-standard-1-3-0)
- Matrix download: [filtered HDF5](https://s3-us-west-2.amazonaws.com/10x.files/samples/cell/1M_neurons/1M_neurons_filtered_gene_bc_matrices_h5.h5)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
path = fetch(
    "1M_neurons_filtered_gene_bc_matrices_h5.h5",
    "https://s3-us-west-2.amazonaws.com/10x.files/samples/cell/1M_neurons/1M_neurons_filtered_gene_bc_matrices_h5.h5",
)

# Scanpy materializes the complete matrix before slicing. For bounded runs,
# read only selected CSC columns from the original 10x HDF5 file.
def read_10x_h5_sample(path, max_cells, random_state=0):
    if max_cells <= 0:
        return sc.read_10x_h5(path)
    import anndata as ad
    import h5py
    import pandas as pd
    from scipy import sparse

    with h5py.File(path, "r") as handle:
        matrix = handle["matrix"] if "matrix" in handle else handle[next(iter(handle))]
        n_features, n_cells = map(int, matrix["shape"][:])
        selected = np.sort(
            np.random.default_rng(random_state).choice(
                n_cells, size=min(max_cells, n_cells), replace=False
            )
        )
        indptr_all = matrix["indptr"][:]
        lengths = indptr_all[selected + 1] - indptr_all[selected]
        indptr = np.r_[0, np.cumsum(lengths)].astype(indptr_all.dtype)
        data = np.concatenate(
            [matrix["data"][indptr_all[i] : indptr_all[i + 1]] for i in selected]
        )
        indices = np.concatenate(
            [matrix["indices"][indptr_all[i] : indptr_all[i + 1]] for i in selected]
        )
        counts = sparse.csc_matrix((data, indices, indptr), shape=(n_features, len(selected)))
        decode = lambda values: np.asarray(values).astype("U")
        barcodes = decode(matrix["barcodes"][selected])
        if "features" in matrix:  # Cell Ranger v3
            features = matrix["features"]
            names = decode(features["name"][:])
            gene_ids = decode(features["id"][:])
            feature_types = decode(features["feature_type"][:])
        else:  # Cell Ranger v2 (the published 1.3M-neuron matrix)
            names = decode(matrix["gene_names"][:])
            gene_ids = decode(matrix["genes"][:])
            feature_types = np.repeat("Gene Expression", n_features)

    obs = pd.DataFrame(index=barcodes)
    var = pd.DataFrame(
        {"gene_ids": gene_ids, "feature_types": feature_types}, index=names
    )
    return ad.AnnData(X=counts.T.tocsr(), obs=obs, var=var)

max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", "20000"))
adata = read_10x_h5_sample(path, max_cells, RANDOM_STATE)
adata.var_names_make_unique()
adata.obs["batch"] = adata.obs_names.str.rsplit("-", n=1).str[-1].astype(str)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.layers["counts"] = adata.X.copy()
adata


## Linear autoencoder and OT


In [ ]:
adata = scb.pp.autoencoder(
    adata,
    input_key="counts",
    out_key="X_ae",
    batch_key="batch",
    n_top_genes=2000,
    latent_dim=30,
    max_epochs=AE_EPOCHS,
    early_stop_patience=5,
    random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata,
    obsm_key="X_ae",
    batch_key="batch",
    out_key="X_ot",
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


In [ ]:
sc.pp.neighbors(adata, use_rep="X_ot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.tl.leiden(adata, resolution=0.8, key_added="leiden_X_ot", random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["batch", "leiden_X_ot"])
